# Aiki-XP — Colab quickstart

A zero-install tour through the public API for the paper *Aiki-XP: leakage-controlled multimodal prediction of within-species relative protein expression at pan-bacterial scale* (Tien, Sharma Meda, Shastry, Mysore, 2026).

**What this notebook does, in order:**
1. Reproduces the paper's headline Tier D ρ_nc = 0.590 ± 0.012 on a fresh 5,000-gene random sample (no GPU needed).
2. Pulls the held-out CV prediction for a specific gene (`NP_417556.2` in *E. coli* K12) and compares to its PaxDb/Abele ground truth.
3. Scores a novel protein end-to-end through Tier D (9/9 modalities, GPU path on Modal).
4. Shows the per-species calibration scatter for *E. coli* K12 (2,785 test genes).

Runs in <3 minutes on Colab's free CPU runtime. Every step hits a public Modal endpoint — no credentials, no local GPU, no data download.

Links:
- **Paper + data:** https://doi.org/10.5281/zenodo.19639621
- **Code:** https://github.com/aikium-public/aiki-xp
- **Live demo:** https://aikium--aikixp-tier-a-landing-page.modal.run
- **Docker:** `ghcr.io/aikium-public/aiki-xp:inference`

In [ ]:
# Standard scientific Python — already on Colab
import requests, numpy as np, pandas as pd
from scipy.stats import spearmanr
import matplotlib.pyplot as plt

LANDING     = 'https://aikium--aikixp-tier-a-landing-page.modal.run'
LOOKUP_GENE = 'https://aikium--aikixp-tier-a-lookup-gene.modal.run'
TIER_D      = 'https://aikium--aikixp-tier-d-predict-tier-d-endpoint.modal.run'

## 1. Reproduce the paper's rho_nc = 0.590 ± 0.012 on a fresh sample

In [ ]:
# Draw a random 5K-gene sample from the 244,002 held-out CV predictions.
r = requests.post(f'{LANDING}/sample_lookup',
                  json={'n': 5000, 'seed': 42}, timeout=60).json()
rows = pd.DataFrame(r['rows'])
print(f"n = {len(rows)}, folds = {sorted(rows['cv_fold'].unique())}, "
      f"mega rate = {rows['is_mega'].mean():.2%}")
rows.head(3)

In [ ]:
# Per-fold mean Spearman rho (the manuscript's reporting convention)
PAPER = {
    'tier_a':      (0.5825, 0.0132, 0.5182, 0.0111),
    'tier_b':      (0.5929, 0.0115, 0.5312, 0.0135),
    'tier_b_plus': (0.6035, 0.0130, 0.5428, 0.0128),
    'tier_c':      (0.6501, 0.0105, 0.5747, 0.0113),
    'tier_d':      (0.6675, 0.0139, 0.5904, 0.0121),
}

def per_fold_rho(df, pred_col, mask=None):
    d = df if mask is None else df[mask]
    rhos = [spearmanr(g['true_expression'], g[pred_col]).statistic
            for _, g in d.groupby('cv_fold') if len(g) >= 2]
    return np.mean(rhos), np.std(rhos)

print(f"{'Tier':7s}  {'rho_ov (yours)':>18s}  {'rho_nm (yours)':>18s}  {'paper rho_nm':>18s}")
for tier, (pov, pov_sd, pnm, pnm_sd) in PAPER.items():
    ov_mean, ov_sd = per_fold_rho(rows, tier)
    nm_mean, nm_sd = per_fold_rho(rows, tier, ~rows['is_mega'])
    print(f"  {tier:7s}  {ov_mean:.3f} ± {ov_sd:.3f}    {nm_mean:.3f} ± {nm_sd:.3f}    "
          f"{pnm:.3f} ± {pnm_sd:.3f}")

You should see per-fold means within ~0.03 of the paper's numbers for all five tiers — the remaining gap is sampling noise at n=5,000 vs the paper's n=244K. Re-run with a different seed and confirm.

## 2. Held-out CV prediction for a specific gene (NP_417556.2, hybF)

In [ ]:
# hybF is a hydrogenase accessory protein in E. coli K12
r = requests.post(LOOKUP_GENE,
                  json={'gene_ids': ['Escherichia_coli_K12|NP_417556.2']}).json()
gene = r['results'][0]
print(f"Gene:         {gene['gene_id']}")
print(f"Species:      {gene['species']}")
print(f"is_mega:      {gene['is_mega']}    (non-mega = true test case)")
print(f"cv_fold:      {gene['cv_fold']}    (this fold held the gene out)")
print(f"PaxDb truth:  {gene['true_expression']:+.3f}")
print(f"Tier A CV:    {gene['tier_a']:+.3f}")
print(f"Tier B CV:    {gene['tier_b']:+.3f}")
print(f"Tier B+ CV:   {gene['tier_b_plus']:+.3f}")
print(f"Tier C CV:    {gene['tier_c']:+.3f}")
print(f"Tier D CV:    {gene['tier_d']:+.3f}")

## 3. Score a novel protein end-to-end through Tier D (9/9 modalities on GPU)

Warms the container on first call (~90 s cold, ~15 s warm). The protein below is NP_415772.1 (ompA-like outer-membrane protein, high expresser).

In [ ]:
import time
payload = {
    'proteins': ['MKKLTVAALAVTTLLSGSAFAHEAGEFFMRAGSATVRPTEGAGGTLGSLGGFSVTNNTQLGLTFTYMATDNIGVELLAATPFRHKIGTRATGDIATVHHLPPTLMAQWYFGDASSKFRPYVGAGINYTTFFDNGFNDHGKEAGLSDLSLKDSWGAAGQVGVDYLINRDWLVNMSVWYMDIDTTANYKLGGAQQHDSVRLDPWVFMFSAGYRF'],
    'cds':      ['ATGAAAAAGTTAACAGTGGCGGCTTTGGCAGTAACAACTCTTCTCTCTGGCAGTGCCTTTGCGCATGAAGCAGGCGAATTTTTTATGCGTGCAGGTTCTGCAACCGTACGTCCAACAGAAGGTGCTGGTGGTACGTTAGGAAGTCTGGGTGGATTCAGCGTGACCAATAACACGCAACTGGGCCTTACGTTTACTTATATGGCGACCGACAACATTGGTGTGGAATTACTGGCAGCGACGCCGTTCCGCCATAAAATCGGCACCCGGGCGACCGGCGATATTGCAACCGTTCATCATCTGCCACCAACACTGATGGCGCAGTGGTATTTTGGTGATGCCAGCAGCAAATTCCGTCCTTACGTTGGGGCAGGTATTAACTACACCACCTTCTTTGATAATGGATTTAACGATCATGGCAAAGAGGCAGGGCTTTCCGATCTCAGTCTGAAAGATTCCTGGGGAGCTGCCGGGCAGGTGGGGGTTGATTATCTGATTAACCGTGACTGGTTGGTTAACATGTCAGTGTGGTACATGGATATCGATACCACCGCCAATTATAAGCTGGGCGGTGCACAGCAACACGATAGCGTACGCCTCGATCCGTGGGTGTTTATGTTCTCAGCAGGATATCGTTTTTAA'],
    'host': 'NC_000913.3',
    'mode': 'native',
    'anchor': 'lacZ',
}
t0 = time.time()
pred = requests.post(TIER_D, json=payload, timeout=1200,
                     allow_redirects=True).json()
print(f"Tier D predicted in {time.time()-t0:.1f}s")
print(f"Modalities filled: {pred['modalities_filled']}")
print(f"Predicted expression (z-score): {pred['predictions'][0]['predicted_expression']:+.3f}")
print(f"(positive z = above the E. coli K12 mean; the ompA-family sequence above is expected to predict well above zero)")

## 4. Per-species calibration scatter for *E. coli* K12

In [ ]:
r = requests.post(f'{LANDING}/species_scatter',
                  json={'species_keys': ['Escherichia_coli_K12', '1006000', '1006004']}).json()
pts = pd.DataFrame(r['points'])
print(f"E. coli K12: {r['n']} test-split genes, {r['n_nonmega']} non-mega")
print(f"rho_overall = {r['rho_overall']:.3f}   rho_non_mega = {r['rho_nonmega']:.3f}")

fig, ax = plt.subplots(figsize=(6, 6))
nm = pts[~pts['is_mega']]; m = pts[pts['is_mega']]
ax.scatter(nm['true'], nm['tier_d'], s=5, alpha=0.5, label=f'non-mega (n={len(nm)})', c='#1E4A8A')
ax.scatter(m['true'],  m['tier_d'],  s=5, alpha=0.5, label=f'mega (n={len(m)})',     c='#C9A961')
ax.plot([-4, 4], [-4, 4], 'k--', alpha=0.3, lw=1, label='y = x')
ax.set_xlabel('PaxDb / Abele z-score (truth)')
ax.set_ylabel('Aiki-XP Tier D (held-out CV)')
ax.set_xlim(-4, 4); ax.set_ylim(-4, 4); ax.set_aspect('equal')
ax.legend(loc='upper left', fontsize=9)
ax.set_title(f'E. coli K12 calibration — rho_nm = {r["rho_nonmega"]:.3f}')
plt.tight_layout(); plt.show()

## What next

- Pick any of the 1,831 cached bacterial hosts by swapping `host=` (use `{LANDING}/hosts.json` to browse).
- Try the other tiers (`/predict_tier_b_endpoint`, `/predict_tier_b_plus_endpoint`, `/predict_tier_c_endpoint`) — same payload shape.
- Download the full 28 GB data + weights deposit from Zenodo and retrain: `python -m zenodo_get 10.5281/zenodo.19639621`.
- Read the paper's Methods for the leakage-control argument (gene-operon split, MMseqs2 cluster-level holdout).

Issues / feedback: https://github.com/aikium-public/aiki-xp/issues